**Setup:** download `data/zip/ecommerce.zip` from the DA Launchpad Datasets page and unzip it in the same folder as this notebook, so that paths like `ecommerce/<table>.csv` work. Needs pandas (and matplotlib/seaborn for charts).

# Ecommerce sales: EDA with joins

This notebook joins orders, order items, products and customers. It checks the row count after each join, then looks at monthly revenue, returns by category, and Prime versus non-Prime buyers.

Put the CSV files for this dataset in a folder named `ecommerce` next to this notebook. The site zip has one folder per dataset. Run the cells from top to bottom.

In [ ]:
import pandas as pd

orders = pd.read_csv("ecommerce/orders.csv").drop_duplicates(subset="order_id")
items = pd.read_csv("ecommerce/order_items.csv")
products = pd.read_csv("ecommerce/products.csv")
customers = pd.read_csv("ecommerce/customers.csv").drop_duplicates(subset="customer_id")
print("orders:", len(orders), "| items:", len(items), "| products:", len(products), "| customers:", len(customers))

## Step 1: Check the grain

One row in `order_items` is one product line in one order. One row in `orders` is one order. Confirm the keys are unique before you join.

In [ ]:
print("items with a duplicate order_item_id:", items["order_item_id"].duplicated().sum())
print("products with a duplicate product_id:", products["product_id"].duplicated().sum())

## Step 2: Parse the order dates

Some order dates are `DD-MM-YYYY` and some are ISO.

In [ ]:
iso = pd.to_datetime(orders["order_date"], format="%Y-%m-%d", errors="coerce")
dmy = pd.to_datetime(orders["order_date"], format="%d-%m-%Y", errors="coerce")
orders["order_dt"] = iso.combine_first(dmy)
print("unparsed order dates:", orders["order_dt"].isna().sum())

## Step 3: Join the tables

Join one table at a time. Check the row count after each join. A many-to-one join must not add rows.

In [ ]:
lines = items.merge(orders[["order_id", "customer_id", "order_dt", "order_status"]], on="order_id", how="inner")
print("after items to orders:", len(lines), "(items had", len(items), "rows)")
lines = lines.merge(products[["product_id", "category"]], on="product_id", how="left")
print("after products:", len(lines))
lines = lines.merge(customers[["customer_id", "is_prime"]], on="customer_id", how="left")
print("after customers:", len(lines))

## Step 4: Monthly revenue

Count only delivered orders. Clean the product category text first, because some values have extra spaces or capitals.

In [ ]:
lines["category"] = lines["category"].str.strip().str.title()
delivered = lines[lines["order_status"] == "Delivered"].copy()
delivered["month"] = delivered["order_dt"].dt.to_period("M")
monthly = delivered.groupby("month")["line_total"].sum().round(2)
monthly.head(12)

## Step 5: Return rate by category

Returns link to an order item. Count the returned items and divide by the items sold.

In [ ]:
returns = pd.read_csv("ecommerce/returns.csv")
returned_items = set(returns["order_item_id"])
lines["returned"] = lines["order_item_id"].isin(returned_items)
rate = lines.groupby("category").agg(items_sold=("order_item_id", "size"), returned=("returned", "sum"))
rate["return_rate_pct"] = (rate["returned"] / rate["items_sold"] * 100).round(2)
rate.sort_values("return_rate_pct", ascending=False)

## Step 6: Prime versus non-Prime

Compare order value and order count per customer. Prime members place more orders, so compare averages, not totals.

In [ ]:
per_order = lines.groupby(["is_prime", "order_id"])["line_total"].sum().reset_index()
summary = per_order.groupby("is_prime")["line_total"].agg(["count", "mean"]).round(2)
summary.columns = ["orders", "avg_order_value"]
summary

## Next questions

- Does the return rate change with the sales channel?
- Is the Prime gap the same in every month?
- Which category has the most revenue but a high return rate?